In [1]:
import jax
import jax.numpy as jnp
from functools import partial
from function_builders import construct_jit_stack_function, construct_jit_sum_function, construct_jit_functions_from_intensity_matrix

jax.config.update('jax_enable_x64', True)

In [2]:
@jax.jit
def mu_1(t, u):
    return jnp.exp(-0.05 * t + u * 0) * 0.25

In [3]:
u = jnp.arange(0, 10, 0.1, dtype=jnp.float64)

In [200]:
@jax.jit
def update_p_j_static(p_j, outflow, inflow, step_size):
    dp_j = jnp.diff(p_j, axis=-1)
    
    outflow_integral = jnp.cumulative_sum(outflow * dp_j, axis=-1, include_initial=True)
    inflow_integral = jnp.sum(inflow * dp_j, axis=(-2, -1), keepdims=True)
    inflow_integral = jnp.squeeze(inflow_integral, axis=2)
    
    p_j = p_j + step_size * (inflow_integral - outflow_integral)
    p_j = jnp.roll(p_j, shift=1, axis=-1)
    p_j = p_j.at[..., 0].set(0)
    
    return p_j

@jax.jit
def update_p_j_static_first(p_j, outflow, inflow, step_size):
    outflow_integral = jnp.pad(outflow, ((0,0), (1,0)))
    inflow_integral = jnp.pad(inflow, ((0,0), (0,0), (1,0)))[:,0,:]
    p_j = p_j + step_size * (inflow_integral - outflow_integral)
    p_j = p_j.at[..., 0].set(0)
    
    return p_j

In [225]:
@partial(jax.jit, static_argnames=['outflow', 'inflow', 'step_size'])
def update_p_j_scan(init_p_j, p_j_0, duration_grid, outflow, inflow, step_size):
    
    def update_p_j(p_j, t):
        outflow_vec = outflow(t, duration_grid)
        inflow_vec = inflow(t, duration_grid)
        p_j = update_p_j_static(p_j, outflow_vec, inflow_vec, step_size)
        
        return p_j, p_j
    
    _, p_j = jax.lax.scan(update_p_j, init_p_j, duration_grid)
    
    p_j = jnp.swapaxes(p_j, 0, 1)
    
    init_p_j = jnp.expand_dims(init_p_j, axis=1)
    p_j_0 = jnp.expand_dims(p_j_0, axis=1)
    return jnp.concatenate([p_j_0, init_p_j, p_j], axis=1)

@partial(jax.jit, static_argnames=['outflow', 'inflow', 'step_size', 'n_states'])
def solve(duration_grid, outflow, inflow, step_size, n_states):
    p_j = jnp.zeros((n_states, duration_grid.shape[0] + 1), dtype=jnp.float64)
    ones = jnp.ones((duration_grid.shape[0] + 1), dtype=jnp.float64)
    
    p_j = p_j.at[0,:].set(ones)
    
    outflow_vec = outflow(0, duration_grid)
    inflow_vec = inflow(0, duration_grid)
    
    p_j_init = update_p_j_static_first(p_j, outflow_vec, inflow_vec, step_size)
    return update_p_j_scan(p_j_init, p_j, duration_grid, outflow, inflow, step_size)

In [260]:
intensity_matrix = [
    [None, mu_1, mu_1, mu_1, None],
    [None, None, mu_1, mu_1, mu_1],
    [None, None, None, None, None],
    [None, None, None, None, None],
    [None, None, None, None, None]
]

step_size = 1 / 12
u_grid = jnp.arange(step_size, 30 + step_size, step_size, dtype=jnp.float64)
outflow, inflow = construct_jit_functions_from_intensity_matrix(intensity_matrix)

In [265]:
%timeit -r 10 -n 10 solve(u_grid, outflow, inflow, step_size, 5).block_until_ready()

7.66 ms ± 722 μs per loop (mean ± std. dev. of 10 runs, 10 loops each)


In [178]:
out

Array([[0.  , 0.5 , 0.5 , ..., 0.5 , 0.5 , 0.5 ],
       [0.  , 0.25, 0.25, ..., 0.25, 0.25, 0.25],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ]], dtype=float64)

In [175]:
inf

Array([[0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.25, 0.25, ..., 0.25, 0.25, 0.25],
       [0.  , 0.25, 0.25, ..., 0.25, 0.25, 0.25]], dtype=float64)

In [ ]:
p * (jnp.pad(inf, ((0,0), (0,0), (1,0))))[:,0,:] - jnp.pad(out, ((0,0), (1,0)))

Array([[ 0.  , -0.5 , -0.5 , ..., -0.5 , -0.5 , -0.5 ],
       [ 0.  ,  0.25,  0.25, ...,  0.25,  0.25,  0.25],
       [ 0.  ,  0.25,  0.25, ...,  0.25,  0.25,  0.25]], dtype=float64)

In [70]:
@partial(jax.jit, static_argnames=['f', 'step_size'])
def update_p_j_scan(init_p_j, p_j_0, duration_grid, f, step_size):
    
    def update_p_j(p_j, t):    
        dp_j = jnp.diff(p_j, axis=0)
        mu_j_dot = f(t, duration_grid)
        
        rhs_int_1 = jnp.cumulative_sum(mu_j_dot * dp_j, include_initial=True)
        
        p_j = p_j - step_size * rhs_int_1
        p_j = jnp.roll(p_j, shift=1, axis=0)
        p_j = p_j.at[0].set(0)
        
        return p_j, p_j
    
    _, p_j = jax.lax.scan(update_p_j, init_p_j, duration_grid)
    
    return jnp.concatenate([jnp.expand_dims(p_j_0, axis=0), jnp.expand_dims(init_p_j, axis=0), p_j], axis=0)

def solve(u_grid, f, step_size):
    p_j = jnp.ones(u_grid.shape[0] + 1, dtype=jnp.float64)
    
    mu_j_dot = f(0, u_grid)
    p_j_init = update_p_j_static_first(p_j, mu_j_dot, step_size)
    
    return update_p_j_scan(p_j_init, p_j, u_grid, f, step_size)

In [ ]:
jnp.cumulative_sum()

72000

In [85]:
j = solve(u_grid, mu_1, step_size)

In [68]:
jnp.expand_dims(j_0, axis=0).shape

(1, 10001)

In [79]:
j.shape

(2002, 2001)

In [368]:
jnp.exp(0.25 * (jnp.exp(-0.05) - 1) / 0.05)

Array(0.78360291, dtype=float64, weak_type=True)